In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import umap.umap_ as umap
import tqdm.auto as tqdm
import typing

In [ ]:
import seaborn as sns

In [ ]:
import networkx as nx


In [ ]:
data = np.array((
    (56, 48, 12, 15, 8, 26, 35, 11, 20, 16, 25, 8),
    (37, 52, 7, 12, 16, 65, 33, 17, 14, 19, 9, 10),
    (48, 47, 10, 29, 15, 27, 28, 19, 28, 13, 9, 12),
    (33, 38, 23, 49, 34, 21, 13, 23, 27, 9, 7, 5),
    (37,21, 39, 42, 43, 16, 17, 24, 14, 14, 8, 5),
    (51, 28, 62, 34, 40, 10, 7, 25, 9, 8, 6, 3),
))
headers = ('LFI', 'EELV', 'PS', 'ENS', 'LR', 'RN')
print(f"{data.shape=}")

In [ ]:
similarity_matrix = cosine_similarity(data)
print(f"{similarity_matrix.shape=}")


In [ ]:
G = nx.Graph()

# Add nodes
for i in range(data.shape[0]):
    G.add_node(i, label=headers[i])

# Add edges with weights based on similarity (threshold for visibility)
threshold = 0.5  # Adjust for clarity in the graph
for i in range(similarity_matrix.shape[1]):
    for j in range(i + 1, similarity_matrix.shape[0]):
        if similarity_matrix[j, i] > threshold:
            G.add_edge(i, j, weight=similarity_matrix[j, i])

# Draw the graph
pos = nx.spring_layout(G, weight='weight')  # Position nodes using the spring layout

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_size=700, node_color='lightblue')

# Draw edges with transparency based on weight
edges = G.edges(data=True)
nx.draw_networkx_edges(
    G, pos, edgelist=[(u, v) for u, v, d in edges],
    alpha=[d['weight'] for _, _, d in edges]
)

# Draw labels
labels = nx.get_node_attributes(G, 'label')
nx.draw_networkx_labels(G, pos, labels, font_size=10)

# Add edge labels for weights (optional)
edge_labels = {(u, v): f"{d['weight']:.2f}" for u, v, d in edges}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)

# Show the plot
plt.title("Proximity Graph Based on Column Similarity")
plt.axis('off')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

for i in range(similarity_matrix.shape[0]):
    similarity_matrix[i, i] = np.nan
    
sns.heatmap(similarity_matrix, annot=True, fmt=".2f", cmap="viridis", cbar_kws={'label': 'Similarity'})
plt.title("Heatmap of Column Similarity")
plt.xlabel("Columns")
plt.ylabel("Columns")
plt.xticks(ticks=np.arange(data.shape[0])+0.5, labels=[headers[i] for i in range(data.shape[0])], rotation=45)
plt.yticks(ticks=np.arange(data.shape[0])+0.5, labels=[headers[i] for i in range(data.shape[0])])
plt.show()


In [ ]:
reducer = umap.UMAP(n_components=2, random_state=42)
embedded = reducer.fit_transform(data)



In [ ]:
def plot2d(embedded):
    plt.figure(figsize=(8, 6))
    for i, (x, y) in enumerate(embedded):
        plt.scatter(x, y, label=headers[i], s=100)
        plt.text(x + 0.02, y + 0.02, headers[i], fontsize=9)
    
    plt.title("2D Projection of Columns")
    plt.xlabel("Dimension 1")
    plt.ylabel("Dimension 2")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
plt.show()

In [ ]:
plot2d(embedded)

In [ ]:
def monkey_step(reference: np.ndarray) -> typing.Tuple[np.ndarray, float]:
    xy = np.random.uniform(-1, 1, size=(reference.shape[0], 2))
    distance = cosine_similarity(xy)
    return xy, np.nansum((reference - distance) ** 2)
    

In [ ]:
steps = 1000000
epochs = []
distance_max = np.inf
for i in tqdm.trange(1, steps):
    candidate, distance = monkey_step(similarity_matrix)

    if distance < distance_max:
        distance_max = distance
        winner = candidate
        epochs.append((i, distance))

epochs = np.array(epochs)


In [ ]:
plt.figure()
plt.plot(epochs[:,0], epochs[:,1])
plt.xlabel('iteration')
plt.ylabel('distance')
plt.xscale('log')
plt.yscale('log')
plt.grid()
plt.show()
plt.close()


In [ ]:
plot2d(winner)
